# Stage 2 Kaggle Smoke Notebook

This notebook is the Kaggle smoke path for Stage 2. Attach the dataset in Kaggle, run Stage 1 first, then run all cells here.

It will auto-clone the repository if needed, resolve the dataset mount, pick a Stage 1 checkpoint, build the latent dataset, run the Stage 2 smoke trainer, and print artifact status.

## Parameters

In [ ]:
# Parameters
HARDWARE_PROFILE = "auto"   # options: "auto", "p100", "t4x2"
RUN_MODE = "first"          # options: "first" or "resume"
REPO_URL = "https://github.com/HoangNguyennnnnnn/DoAn_backup.git"
DATASET_SLUG = "balraj98/modelnet40-princeton-3d-object-dataset"
DATASET_ROOT_OVERRIDE = ""   # optional explicit /kaggle/input path
STAGE1_CHECKPOINT = ""       # optional explicit Stage 1 checkpoint path
RESUME_CHECKPOINT_PATH = ""   # leave empty to use latest_step.ckpt in resume mode
RUN_ID = "kaggle-stage2-smoke"
LATENT_SCHEMA_VERSION = "stage2-latent-v1"
LATENT_PREP_CHECKPOINT_PREFERENCE = "latest_step,best,interrupt,latest"
SMOKE_NUM_SAMPLES = 4
SMOKE_BATCH_SIZE = 8

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path


def _run(command, cwd=None):
    printable = " ".join(str(part) for part in command)
    print(f"$ {printable}")
    subprocess.run([str(part) for part in command], cwd=str(cwd) if cwd else None, check=True)


def _ensure_repo_root(repo_url: str) -> Path:
    repo_name = Path(repo_url.rstrip("/")).name.replace(".git", "")
    candidates = [
        Path("/kaggle/working/DoAn_backup"),
        Path("/kaggle/working") / repo_name,
        Path.cwd(),
    ]

    for candidate in candidates:
        if (candidate / "scripts").is_dir() and (candidate / "configs").is_dir():
            if (candidate / ".git").exists():
                subprocess.run(["git", "pull", "--ff-only"], cwd=str(candidate), check=True)
            return candidate.resolve()

    target = Path("/kaggle/working/DoAn_backup")
    if target.exists():
        shutil.rmtree(target)
    _run(["git", "clone", repo_url, str(target)])
    return target.resolve()


def _find_dataset_root(dataset_slug: str, explicit_root: str = "") -> Path:
    input_root = Path("/kaggle/input")
    candidates = []
    if explicit_root.strip():
        candidates.append(Path(explicit_root.strip()))
    candidates.append(input_root / dataset_slug.split("/")[-1])

    if input_root.exists():
        candidates.extend(child for child in sorted(input_root.iterdir()) if child.is_dir())

    seen = set()
    for candidate in candidates:
        resolved = candidate.resolve() if candidate.exists() else candidate
        key = str(resolved)
        if key in seen:
            continue
        seen.add(key)
        if resolved.exists() and resolved.is_dir():
            if any(resolved.rglob("*.off")) or any(resolved.rglob("*.obj")):
                return resolved

    mounted = [child.name for child in sorted(input_root.iterdir()) if child.is_dir()] if input_root.exists() else []
    raise RuntimeError(
        "Dataset root missing. Attach the Kaggle dataset in the notebook UI before running Stage 2. "
        f"Checked candidates: {[str(candidate) for candidate in candidates]}. "
        f"Mounted /kaggle/input folders: {mounted or ['none']}"
    )


def _resolve_hardware_config(profile: str) -> Path:
    profile = profile.strip().lower()
    if profile == "p100":
        return Path("configs/hardware_p100.yaml")
    if profile == "t4x2":
        return Path("configs/hardware_t4x2.yaml")
    if profile != "auto":
        raise ValueError(f"Unsupported HARDWARE_PROFILE: {profile}")

    try:
        import torch

        if torch.cuda.is_available():
            gpu_name = torch.cuda.get_device_name(0).lower()
            if "t4" in gpu_name and torch.cuda.device_count() >= 2:
                return Path("configs/hardware_t4x2.yaml")
    except Exception:
        pass

    return Path("configs/hardware_p100.yaml")


def _normalize_stage1_config(repo_root: Path) -> None:
    config_path = repo_root / "configs" / "train_stage1.yaml"
    if not config_path.exists():
        return

    text = config_path.read_text(encoding="utf-8")
    if "${GRAD_ACC_STEPS}" in text:
        config_path.write_text(text.replace("${GRAD_ACC_STEPS}", "1"), encoding="utf-8")
        print(f"Normalized placeholder in {config_path}")


def _pick_stage1_checkpoint(checkpoint_dir: Path, explicit_path: str = "") -> Path:
    if explicit_path.strip():
        candidate = Path(explicit_path.strip())
        if candidate.exists():
            return candidate.resolve()
        raise FileNotFoundError(f"Stage 1 checkpoint not found: {candidate}")

    for name in ["best.ckpt", "latest.ckpt", "latest_step.ckpt", "interrupt.ckpt"]:
        candidate = checkpoint_dir / name
        if candidate.exists():
            return candidate.resolve()

    raise FileNotFoundError(
        f"No Stage 1 checkpoint found in {checkpoint_dir}. Run Stage 1 first or set STAGE1_CHECKPOINT."
    )

In [ ]:
repo_root = _ensure_repo_root(REPO_URL)
os.chdir(repo_root)
print(f"Project root: {repo_root}")

_normalize_stage1_config(repo_root)

dataset_root = _find_dataset_root(DATASET_SLUG, DATASET_ROOT_OVERRIDE)
hardware_config = _resolve_hardware_config(HARDWARE_PROFILE)
output_root = Path("/kaggle/working")
checkpoint_dir = output_root / "checkpoints"

os.environ["DATASET_SLUG"] = DATASET_SLUG
os.environ["DATASET_ROOT"] = str(dataset_root)
os.environ["OUTPUT_ROOT"] = str(output_root)
os.environ["TRAIN_CONFIG"] = "configs/train_stage2.yaml"
os.environ["DATA_CONFIG"] = "configs/data_stage2.yaml"
os.environ["HARDWARE_CONFIG"] = str(hardware_config)
os.environ["RUN_ID"] = RUN_ID

stage1_checkpoint = _pick_stage1_checkpoint(checkpoint_dir, STAGE1_CHECKPOINT)
os.environ["STAGE1_CHECKPOINT_PATH"] = str(stage1_checkpoint)

if RUN_MODE not in {"first", "resume"}:
    raise ValueError(f"Unsupported RUN_MODE: {RUN_MODE}")

if RUN_MODE == "resume":
    resume_path = RESUME_CHECKPOINT_PATH.strip() or str(checkpoint_dir / "latest_step.ckpt")
    os.environ["RESUME_CHECKPOINT_PATH"] = resume_path

print(f"Dataset root: {dataset_root}")
print(f"Hardware config: {hardware_config}")
print(f"Stage 1 checkpoint: {stage1_checkpoint}")
if RUN_MODE == "resume":
    print(f"Resume checkpoint: {os.environ['RESUME_CHECKPOINT_PATH']}")

_run(["bash", "scripts/kaggle_bootstrap.sh"], cwd=repo_root)

In [ ]:
build_cmd = [
    "python",
    "scripts/build_latent_dataset.py",
    "--train-config",
    "configs/train_stage1.yaml",
    "--data-config",
    os.environ["DATA_CONFIG"],
    "--dataset-root",
    os.environ["DATASET_ROOT"],
    "--output-root",
    os.environ["OUTPUT_ROOT"],
    "--checkpoint-path",
    os.environ["STAGE1_CHECKPOINT_PATH"],
    "--checkpoint-preference",
    LATENT_PREP_CHECKPOINT_PREFERENCE,
    "--split",
    "both",
    "--batch-size",
    str(SMOKE_BATCH_SIZE),
    "--latent-schema-version",
    LATENT_SCHEMA_VERSION,
    "--device",
    "auto",
    "--enforce-kaggle",
]
_run(build_cmd, cwd=repo_root)

smoke_cmd = [
    "python",
    "scripts/train_stage2.py",
    "--config",
    os.environ["TRAIN_CONFIG"],
    "--hardware",
    os.environ["HARDWARE_CONFIG"],
    "--data-config",
    os.environ["DATA_CONFIG"],
    "--stage1-checkpoint",
    os.environ["STAGE1_CHECKPOINT_PATH"],
    "--output-root",
    os.environ["OUTPUT_ROOT"],
    "--device",
    "auto",
    "--run-id",
    os.environ["RUN_ID"],
]

if RUN_MODE == "resume":
    smoke_cmd.extend(["--resume-from", os.environ["RESUME_CHECKPOINT_PATH"]])

_run(smoke_cmd, cwd=repo_root)

In [ ]:
log_dir = Path(os.environ["OUTPUT_ROOT"]) / "logs"
latent_manifest_root = Path(os.environ["OUTPUT_ROOT"]) / "cache" / "stage2_latents" / LATENT_SCHEMA_VERSION / "manifests"

print("Checkpoint status:")
for name in ["latest_step.ckpt", "interrupt.ckpt", "latest.ckpt", "best.ckpt", "smoke_latest.ckpt"]:
    path = checkpoint_dir / name
    print(f"  {name}: exists={path.exists()} path={path}")

print("\nLatent manifests:")
for name in ["latent_manifest_train.jsonl", "latent_manifest_test.jsonl"]:
    path = latent_manifest_root / name
    print(f"  {name}: exists={path.exists()} path={path}")

print("\nSmoke reports:")
for name in ["stage2_smoke_summary.json", "stage2_checkpoint_integrity.json", "stage2_smoke_metrics.jsonl"]:
    path = log_dir / name
    print(f"  {name}: exists={path.exists()} path={path}")